# Übung 08: EDA – grafisch (matplotlib, seaborn)

**Übungsnotebook zur Vorbereitung auf LN 1 (Mo. 09.11.2026)** · Cheatsheet: **Kapitel 8 (EDA – grafisch) und 9.3/9.5** und Kapitel 11 (Checklisten)

| | |
|---|---|
| **Themen** | Passende Grafik wählen, objektorientiertes matplotlib-Gerüst, subplots, pandas-Plotting, seaborn (hist, box, violin, count, bar, scatter, reg, pair, heatmap), QQ-Plot, Log-Skala, Interpretation |
| **Format** | wie die Musterprüfungen AP01–AP03: Teil A Konzeptfragen · Teil B Fehleranalyse · Teil C Code schreiben |
| **Zeitbudget** | ca. 230 Minuten (Prüfungstempo: ca. 4.5 Min. pro Punkt) |
| **Lösungen** | keine – alles ist mit dem Cheatsheet bzw. den Kursunterlagen lösbar. Hinweise wie *(→ 4.5)* verweisen auf den Cheatsheet-Abschnitt. |

**Szenario:** Der fiktive Lieferdienst *FoodFlitzer* liefert Essen in Winterthur mit Velos, E-Scootern und Autos. Sie erhalten alle Lieferungen von August bis Mitte Oktober 2026 und sollen die Daten grafisch explorieren.

**So üben Sie prüfungsnah**
- Copilot/KI-Autocomplete ausschalten, wie in der Prüfung (Cheatsheet 1.1).
- Begründungen immer **mit konkretem Bezug zu den Daten** (Spaltennamen, Werte, Zahlen aus dem Output) – sonst gibt es in der Prüfung 0 Punkte.
- Verändern Sie die Ausgangsdaten nicht, sondern arbeiten Sie pro Aufgabe mit eigenen Variablen (z.B. `df_B1`, `df_C2`).
- Am Schluss: *Kernel → Restart & Run All* (Cheatsheet 11.5).

**Bewertung von Grafiken in der Prüfung:** Ohne Titel, Achsenbeschriftung (mit Einheit) bzw. Legende bei mehreren Gruppen gibt es Abzug (Cheatsheet 8, 11.5). Gewöhnen Sie sich an, diese **immer** zu setzen.

## Setup (wird nicht bewertet)

Die folgende Zelle erzeugt die Übungsdaten **synthetisch und reproduzierbar** (fixer Seed) im Unterordner neben diesem Notebook. Sie brauchen den Code nicht zu verstehen und sollen ihn **nicht verändern** – behandeln Sie die erzeugten Dateien so, als hätten Sie sie in der Prüfung erhalten. *(Natürlich dürfen Sie die Dateien im Editor öffnen und ansehen.)*

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Settings
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

# Get current working directory
print(os.getcwd())

In [ ]:
# ---------------------------------------------------------------
# Data generator (do not modify) -> folder 'data_uebung08'
# ---------------------------------------------------------------
import os
import numpy as np
import pandas as pd

rng = np.random.default_rng(808)
D = 'data_uebung08'
os.makedirs(D, exist_ok=True)

districts = ['Altstadt', 'Mattenbach', 'Oberwinterthur', 'Seen', 'Töss', 'Veltheim', 'Wülflingen']
d_w = np.array([0.24, 0.13, 0.17, 0.11, 0.12, 0.12, 0.11])
d_dist = {'Altstadt': 1.4, 'Mattenbach': 2.2, 'Oberwinterthur': 3.0, 'Seen': 3.4, 'Töss': 2.4, 'Veltheim': 2.0, 'Wülflingen': 3.1}

days = pd.date_range('2026-08-01', '2026-10-15', freq='D')
day_factor = 1 + 0.35 * np.isin(days.dayofweek, [4, 5]) + 0.2 * (days.dayofweek == 6)
rain_days = rng.random(len(days)) < 0.25
day_factor = day_factor * np.where(rain_days, 1.45, 1.0) * (1 + 0.004 * np.arange(len(days)))
n_per_day = rng.poisson(75 * day_factor)

rows = []
for di, (d, k) in enumerate(zip(days, n_per_day)):
    for _ in range(k):
        dist_name = districts[rng.choice(len(districts), p=d_w)]
        rider = rng.choice(['Velo', 'E-Scooter', 'Auto'], p=[0.68, 0.22, 0.10])
        km = round(float(rng.gamma(4, d_dist[dist_name] / 4)) + 0.2, 2)
        speed = {'Velo': 15, 'E-Scooter': 19, 'Auto': 17}[rider] * rng.uniform(0.8, 1.15)
        prep = rng.gamma(4, 3)                                 # kitchen time
        minutes = round(prep + km / speed * 60 + (6 if rider == 'Auto' else 2) + rng.normal(0, 3) + (8 if rain_days[di] else 0), 1)
        hour = int(rng.choice([11, 12, 13, 17, 18, 19, 20, 21], p=[0.08, 0.17, 0.10, 0.07, 0.17, 0.22, 0.13, 0.06]))
        value = round(float(np.exp(rng.normal(3.55, 0.45))), 2)
        tip = 0.0 if rng.random() < 0.55 else round(float(min(value * rng.uniform(0.03, 0.15), 25)), 1)
        rating = int(np.clip(np.round(5.2 - 0.045 * minutes + rng.normal(0, 0.7)), 1, 5))
        rows.append((d.strftime('%Y-%m-%d'), hour, dist_name, rider, km, max(minutes, 8.0), value, tip, rating, bool(rain_days[di])))
df = pd.DataFrame(rows, columns=['date', 'hour', 'district', 'rider_type', 'distance_km', 'delivery_min',
                                 'order_value_chf', 'tip_chf', 'rating', 'rain'])
df.insert(0, 'order_id', [f'D{200000 + i}' for i in range(len(df))])
big = rng.choice(len(df), size=12, replace=False)
df.loc[big, 'order_value_chf'] = np.round(rng.uniform(250, 600, 12), 2)    # company catering orders
df.to_csv(f'{D}/deliveries.csv', index=False)
print('Files created:', sorted(os.listdir(D)), '|', len(df), 'Lieferungen')

### Daten laden

`deliveries.csv`: `order_id`, `date`, `hour` (Bestellstunde), `district` (Quartier), `rider_type` (`Velo`, `E-Scooter`, `Auto`), `distance_km`, `delivery_min` (Bestellung bis Übergabe), `order_value_chf`, `tip_chf`, `rating` (1–5), `rain` (Regentag ja/nein)

In [ ]:
# Load the data (do not modify df)
df = pd.read_csv('data_uebung08/deliveries.csv', parse_dates=['date'])
print(df.shape)
df.head()

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnen die Übungsaufgaben — 60 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zu den FoodFlitzer-Daten werden nicht gewertet.*

### A1 (4 Punkte)

Welche Grafik (und welche seaborn- bzw. matplotlib-Funktion) wählen Sie für die folgenden Fragen? Begründen Sie jeweils in einem Halbsatz. *(→ 8.1)*

1. Wie ist der Bestellwert `order_value_chf` verteilt?
2. Unterscheidet sich die Lieferzeit zwischen den drei `rider_type`?
3. Wie hat sich die Anzahl Bestellungen pro Tag seit August entwickelt?
4. Wie hängt die Lieferzeit von der Distanz ab (über 8'000 Punkte)?
5. Welche Quartiere bestellen am häufigsten?
6. Wie hängen alle numerischen Variablen paarweise zusammen?

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

Die folgende Grafik wurde einer Geschäftsleitung präsentiert. Führen Sie die Zelle aus und nennen Sie **drei** konkrete Mängel, die zu Fehlinterpretationen führen können, sowie je eine Verbesserung. *(→ 8.1 «Tipp», 8.2)*

In [ ]:
# Chart presented to the management (do not modify)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
m = df.groupby('rider_type')['delivery_min'].mean()
axes[0].bar(m.index, m.values, color=['red', 'green', 'blue'])
axes[0].set_ylim(24, 32)
axes[0].set_title('Auto ist VIEL langsamer!')
df['district'].value_counts().plot(kind='pie', ax=axes[1])
axes[1].set_ylabel('')
plt.show()

**Ihre Antwort:**

> [Hier eintragen]

### A3 (4 Punkte)

Erstellen Sie nebeneinander je einen **QQ-Plot** (Normalverteilung) für `order_value_chf` und für `np.log(order_value_chf)`. Beschreiben Sie die Form der Punktwolke in beiden Plots und was sie über die Verteilung aussagt (Schiefe, Ränder, Ausreisser). Was würden Sie daraus für die Darstellung des Bestellwerts in einem Histogramm ableiten? *(→ 8.5, 9.3)*

In [ ]:
# Code hier eintragen

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – Fehleranalyse Python-Code (16 Punkte)

*Jeder Code läuft fehlerfrei durch, liefert aber eine irreführende Grafik oder Schlussfolgerung.*

### B1 (8 Punkte)

**Ziel:** Vergleich der **Verteilung** der Lieferzeit zwischen den Fahrzeugtypen.

In [ ]:
# Original code (do not modify this cell)
fig, ax = plt.subplots(figsize=(8, 4))
for rider in ['Velo', 'E-Scooter', 'Auto']:
    ax.hist(df.loc[df['rider_type'] == rider, 'delivery_min'], alpha=0.5, label=rider)
ax.set_title('Lieferzeit')
ax.legend()
plt.show()
print('Fazit: Velo-Lieferungen dauern meistens am längsten (höchste Balken).')

**Aufgabe:** Erklären Sie, warum das Fazit aus dieser Grafik nicht gezogen werden kann (mindestens **zwei** Gründe; Tipp: `value_counts()` von `rider_type`, Bin-Grenzen der drei Histogramme). Erstellen Sie eine korrigierte Darstellung mit **zwei** Teilgrafiken nebeneinander:

1. links ein Histogramm mit **gemeinsamen** Bins und vergleichbarer y-Achse (Dichte statt Anzahl),
2. rechts einen gruppierten Box- oder Violinplot.

Beschriften Sie beide vollständig und formulieren Sie das korrigierte Fazit mit Zahlen (z.B. Mediane). Begründen Sie jede Änderung. *(→ 8.2–8.4)*

**Begründung der Änderungen und korrigiertes Fazit:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code hier eintragen

### B2 (8 Punkte)

**Ziel:** Links der Zusammenhang Distanz → Lieferzeit, rechts die Verteilung der Distanz – beide vollständig beschriftet.

In [ ]:
# Original code (do not modify this cell)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(df['distance_km'], df['delivery_min'], s=40)
plt.title('Distanz vs. Lieferzeit')
plt.xlabel('Distanz (km)')
plt.ylabel('Lieferzeit (min)')
axes[1].hist(df['distance_km'], bins=40)
plt.title('Verteilung der Distanz')
plt.show()
print('Fazit: Die Punktwolke ist ein grosser Klumpen – Distanz und Lieferzeit hängen kaum zusammen.')

**Aufgabe:** Erklären Sie, warum die Beschriftungen nicht dort landen, wo sie hingehören, und warum das Fazit aus dem Scatterplot vorschnell ist. Schreiben Sie den korrigierten Code: korrekte Beschriftung **jeder** Achse über das jeweilige `Axes`-Objekt, eine Darstellung, die das Overplotting reduziert (z.B. Transparenz, Hexbin oder `regplot` mit Trendlinie), und die Ausgabe von Pearson- und Spearman-Korrelation. Formulieren Sie das korrigierte Fazit und begründen Sie jede Änderung. *(→ 8.1–8.3, 9.5)*

**Begründung der Änderungen und korrigiertes Fazit:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code hier eintragen

---
## Teil C – Grafiken erstellen (32 Punkte)

*Erstellen Sie die Grafiken vollständig beschriftet und interpretieren Sie jeweils in 1–2 Sätzen.*

### C1 (4 Punkte)

Erstellen Sie eine Figur mit **zwei** Teilgrafiken nebeneinander für `order_value_chf`: links ein Histogramm (40 Bins) mit vertikalen Linien für **Mittelwert** und **Median** (unterschiedliche Farben, Legende), rechts dasselbe Histogramm mit **logarithmischer x-Achse** (z.B. `log_scale=True` in seaborn). Was zeigen die Lage von Mittelwert und Median und die Log-Darstellung? *(→ 8.2, 8.3, 8.5)*

**Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C2 (4 Punkte)

Erstellen Sie einen gruppierten **Boxplot** der Lieferzeit pro Quartier (`district`). Die Quartiere sollen **nach Median-Lieferzeit sortiert** sein, und die Beschriftung der x-Achse soll die Anzahl Lieferungen enthalten (z.B. `Altstadt\n(n=1951)`). Fügen Sie eine horizontale Linie für die Median-Lieferzeit über alle Lieferungen hinzu. Welches Quartier fällt auf, und womit könnte das zusammenhängen? *(→ 8.4, 6.6)*

**Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C3 (4 Punkte)

Erstellen Sie ein **Liniendiagramm** der Anzahl Lieferungen pro Tag mit einer zweiten Linie für den **gleitenden 7-Tage-Mittelwert**. Markieren Sie Regentage mit Punkten auf der Tageslinie und beschriften Sie den Tag mit den meisten Lieferungen mit `ax.annotate` (Datum und Anzahl). Welche Muster erkennen Sie (Trend, Wochenrhythmus, Wetter)? *(→ 8.2, 5.7 «Zeitreihe glätten»)*

**Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C4 (4 Punkte)

Erstellen Sie eine **Heatmap** der **Spearman**-Korrelationsmatrix aller numerischen Variablen (ohne `hour`), nur das **untere Dreieck**, mit Werten (2 Stellen), divergierender Farbskala von −1 bis +1 und Titel. Nennen Sie die zwei stärksten Zusammenhänge und interpretieren Sie sie inhaltlich. *(→ 8.4)*

**Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C5 (4 Punkte)

Erstellen Sie ein **horizontales Balkendiagramm** des **Medians** von `tip_chf` pro Quartier, absteigend sortiert, mit den Werten an den Balken (`ax.bar_label`). Erstellen Sie daneben ein zweites Balkendiagramm mit dem **Anteil Lieferungen ohne Trinkgeld** pro Quartier in %. Warum ist hier der Median allein wenig aussagekräftig? *(→ 8.2, 8.3)*

**Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C6 (4 Punkte)

Zeigen Sie mit einem **gruppierten Balkendiagramm** (`sns.barplot` mit `hue`) die **mittlere Bewertung** (`rating`) pro Fahrzeugtyp, getrennt nach Regentag ja/nein, mit Fehlerbalken. Beschriften Sie Legende und Achsen auf Deutsch. Ist der Unterschied zwischen Regen- und Trockentagen bei allen Fahrzeugtypen ähnlich? *(→ 8.1, 8.4)*

**Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C7 (4 Punkte)

Erstellen Sie einen **Pairplot** für `distance_km`, `delivery_min`, `order_value_chf` und `rating`, eingefärbt nach `rider_type`, auf einer **Zufallsstichprobe** von 800 Lieferungen (`random_state=42`). Setzen Sie einen Gesamttitel. Warum ist eine Stichprobe hier sinnvoll, und welches Risiko birgt sie? *(→ 8.4 «Tipp»)*

**Begründung und Interpretation:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C8 (4 Punkte)

Erstellen Sie eine 2×2-Übersichtsgrafik («Dashboard») mit `plt.subplots(2, 2)`: (1) Anzahl Bestellungen pro Stunde (Balken), (2) Anteil Fahrzeugtypen (Balken in %, **kein** Kreisdiagramm), (3) Lieferzeit vs. Bewertung (Boxplot von `delivery_min` je `rating`), (4) Bestellwert pro Wochentag (Boxplot, Mo–So sortiert, ohne Ausreisser-Punkte). Jede Teilgrafik mit Titel und Achsenbeschriftungen, dazu ein Gesamttitel; verwenden Sie `plt.tight_layout()`. Formulieren Sie die **eine** wichtigste Erkenntnis für das Management. *(→ 8.3)*

**Wichtigste Erkenntnis:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Libraries
import os, platform, socket
from platform import python_version
from datetime import datetime

# Print system information
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('-----------------------------------')